# 03. HuggingFace 실전 — 직접 만든 Transformer 가 어떻게 LLM 으로 이어지는가

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/glasslego/2026-ml-study/blob/main/notebooks/study_03_transformer/03_huggingface_practical.ipynb)

이전 두 노트북에서 우리가 직접 만든 transformer block 이 실제로는
**HuggingFace 의 한 줄 모델 로드** 와 동일한 구조다. 이 노트북에서는:

1. HuggingFace 생태계 개요 (transformers / datasets / tokenizers / accelerate / hub)
2. 사전학습 모델 한 줄 로드 — 우리가 만든 block 과 같은 모양인지 확인
3. Tokenizer — 텍스트 → 토큰 ID 의 본질
4. 모델 forward 와 출력 구조
5. `pipeline` 한 줄 추론
6. `datasets` 로 데이터 로드 / 전처리
7. `Trainer` API 로 fine-tuning 맛보기
8. 한국어 모델 사용 예 (klue/bert-base, beomi/kcbert, KoBigBird 등)
9. encoder-only(BERT) → decoder-only(GPT) → instruct-tuned LLM (LLaMA/Mistral/Gemma) 흐름
10. 실전 팁 (모델 카드, 라이선스, 양자화, 로컬 추론)

> **CPU 만으로도 실행되도록** 가벼운 모델 (`distilbert-base-uncased`, `sshleifer/tiny-gpt2` 등) 을 사용한다.
> Colab GPU 에서는 더 큰 모델로 바로 바꿔 실행 가능.


In [ ]:
# 공통 실행 환경 준비 (Colab/Local 통합)
from pathlib import Path
import subprocess, sys, os

REPO_URL = 'https://github.com/glasslego/2026-ml-study.git'
REPO_NAME = '2026-ml-study'
NOTEBOOK_SUBDIR = 'notebooks/study_03_transformer'

if 'google.colab' in sys.modules:
    clone_root = Path('/content') / REPO_NAME
    if not clone_root.exists():
        subprocess.run(['git', 'clone', REPO_URL, str(clone_root)], check=True)
    target = clone_root / NOTEBOOK_SUBDIR
else:
    target = None
    for c in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (c / NOTEBOOK_SUBDIR).exists():
            target = c / NOTEBOOK_SUBDIR
            break
    if target is None:
        raise FileNotFoundError(f'{NOTEBOOK_SUBDIR} 를 찾지 못했습니다.')

os.chdir(target)
print(f'작업 디렉토리: {target}')

# HuggingFace 패키지 설치 (없으면). Colab 에는 transformers 가 기본 설치되어 있음.
def _ensure(pkg, import_name=None):
    try:
        __import__(import_name or pkg)
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=True)

for p in ['torch', 'transformers', 'datasets', 'accelerate']:
    _ensure(p)


## 1. HuggingFace 생태계 개요

| 라이브러리        | 역할                                                                 |
|-------------------|----------------------------------------------------------------------|
| `transformers`    | 수백 종 사전학습 모델의 통일된 인터페이스 (`from_pretrained`)         |
| `tokenizers`      | 빠른 BPE/WordPiece 등 토크나이저 (Rust 구현)                         |
| `datasets`        | 수천 종 데이터셋의 통일된 로더 (`load_dataset`)                      |
| `accelerate`      | 분산/혼합정밀도 학습 추상화 (`accelerate launch`)                    |
| `peft`            | LoRA / QLoRA 등 파라미터 효율 fine-tuning                            |
| `evaluate`        | 평가 지표 (BLEU, ROUGE, accuracy 등) 통일 인터페이스                 |
| `huggingface_hub` | 모델/데이터셋 공유 (사실상 ML 의 GitHub)                             |

핵심 발상: **모델 = 가중치 + 설정(config) + 토크나이저** 이 세 가지를 표준화해 어떤 모델이든 같은 방법으로 쓰게 한다.


## 2. 사전학습 모델 한 줄 로드 — 우리가 만든 block 과 같은가?

`distilbert-base-uncased` (BERT 의 가벼운 변형, 6층) 를 가져와 내부 모듈을 들여다보자.
이전 노트북에서 만든 `TransformerEncoderBlock` 과 **이름까지 거의 같은 구조** 임을 확인할 수 있다.


In [ ]:
from transformers import AutoModel, AutoTokenizer, AutoConfig
import torch

MODEL_NAME = 'distilbert-base-uncased'

config = AutoConfig.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)

print(f'모델 종류 : {type(model).__name__}')
print(f'층 수      : {config.n_layers}')
print(f'd_model    : {config.dim}')
print(f'head 수    : {config.n_heads}')
print(f'd_ff       : {config.hidden_dim}')
print(f'전체 파라미터: {sum(p.numel() for p in model.parameters()):,}')


In [ ]:
# 첫 번째 transformer 층의 내부를 들여다본다 — attention(MHA), ffn(FFN), layer_norm 이 보인다.
first_layer = model.transformer.layer[0]
print(first_layer)


> 위 구조에서 `MultiHeadSelfAttention` + `FFN` + `LayerNorm` 이 보인다. 이전 노트북에서 우리가 짠
> `TransformerEncoderBlock` 과 정확히 같은 부품의 조합이다. 차이는 가중치가 **수십 GB 의 텍스트로 미리 학습됨** 이라는 것뿐.


## 3. Tokenizer — "텍스트" 가 어떻게 "토큰 ID" 가 되는가

모델이 받는 입력은 정수 ID 다. 텍스트 → ID 변환을 담당하는 게 토크나이저.
대표 알고리즘: **BPE (Byte-Pair Encoding)**, **WordPiece**, **SentencePiece**.

핵심: 자주 쓰이는 단어/조각은 한 토큰, 드문 것은 여러 sub-word 로 쪼개 어휘 크기를 작게 유지.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

text = "Transformers turn text into tokens."
enc = tokenizer(text, return_tensors='pt')
print('입력 텍스트  :', text)
print('input_ids    :', enc['input_ids'])
print('attention_mask:', enc['attention_mask'])
print('토큰 복원    :', tokenizer.convert_ids_to_tokens(enc['input_ids'][0]))


In [ ]:
# sub-word 분해 시연 — 사전에 없는 합성어가 어떻게 쪼개지는지
for w in ['transformer', 'transformers', 'unhappiness', 'unboxing']:
    toks = tokenizer.tokenize(w)
    print(f'{w:>14} -> {toks}')


## 4. 모델 forward — 입력/출력의 의미

`AutoModel` (분류 head 없는 base) 에 토큰을 넣으면 **모든 토큰 위치의 표현 (last_hidden_state)** 이 나온다.
shape 은 `(batch, seq_len, d_model)`.

여기에 task-specific head (분류용 linear, MLM 용 vocab 분류 등) 를 붙이면 BertForSequenceClassification 같은 클래스가 된다.


In [ ]:
model.eval()
with torch.no_grad():
    out = model(**enc)
print('last_hidden_state shape:', out.last_hidden_state.shape, '  (batch, seq_len, d_model)')
print('첫 토큰 ([CLS]) 표현 첫 5차원:', out.last_hidden_state[0, 0, :5])


## 5. `pipeline` — 한 줄 추론

`pipeline` 은 토크나이즈 + 모델 + 후처리를 묶어 "한 줄로 쓸 수 있게" 한 고수준 API.
프로토타이핑/데모에 적합.


In [ ]:
from transformers import pipeline

# 감정 분류 (영어, 가벼움)
sentiment = pipeline('sentiment-analysis', model='distilbert-base-uncased-finetuned-sst-2-english')
for t in ["I love this notebook!", "This is terrible and frustrating."]:
    print(t, '->', sentiment(t)[0])


In [ ]:
# 빈칸 채우기 (masked language modeling) - BERT 류의 본질적 사전학습 task
fill = pipeline('fill-mask', model='distilbert-base-uncased')
for r in fill(f"The capital of France is {fill.tokenizer.mask_token}.")[:3]:
    print(f"{r['score']:.3f}  {r['token_str']}")


## 6. `datasets` — 데이터 로드/전처리

`load_dataset` 한 줄로 표준 데이터셋을 다운로드 + 캐싱. 토크나이즈/전처리는 `.map()` 으로 일괄 처리.

> 네트워크가 없는 환경이면 셀이 실패할 수 있다. 작은 합성 데이터로 fallback 도 보여준다.


In [ ]:
try:
    from datasets import load_dataset
    # IMDb 영화 리뷰 — 작지 않지만 잘 알려진 분류 데이터
    raw = load_dataset('imdb', split='train[:200]+test[:50]')
    print(raw)
except Exception as e:
    print('네트워크 또는 datasets 이슈로 IMDb 로드 실패. 합성 데이터로 대체:', repr(e))
    from datasets import Dataset
    samples = [
        {'text': '정말 좋은 영화였다.', 'label': 1},
        {'text': '시간 낭비였다.', 'label': 0},
        {'text': '배우 연기가 훌륭했다.', 'label': 1},
        {'text': '지루하고 길었다.', 'label': 0},
    ] * 25
    raw = Dataset.from_list(samples)
    print(raw)

# 토크나이즈 (배치 처리)
def tok_fn(batch):
    return tokenizer(batch['text'], padding='max_length', truncation=True, max_length=64)

tokenized = raw.map(tok_fn, batched=True)
print(tokenized.column_names)


## 7. `Trainer` API — Fine-tuning 맛보기

`Trainer` 는 학습 루프, 평가, 로깅, 체크포인트, 분산학습까지 캡슐화한 고수준 학습기.
이전 노트북에서 우리가 직접 짠 `TorchTrainer.fit` 과 같은 일을 더 풍부하게 한다.

여기서는 **CPU 에서도 분 단위로 끝나도록** 매우 작은 fine-tuning 만 시연 (실제 학습은 GPU 권장).


In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np

# 작은 분류 모델 (head 가 자동으로 붙는다)
clf = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

# 데이터 분리
ds = tokenized.train_test_split(test_size=0.2, seed=0)
train_ds = ds['train'].select(range(min(len(ds['train']), 64)))   # CPU 시연용으로 작게
eval_ds  = ds['test'].select(range(min(len(ds['test']), 16)))

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {'accuracy': float((preds == labels).mean())}

args = TrainingArguments(
    output_dir='./hf_demo_out',
    num_train_epochs=1,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=5e-5,
    eval_strategy='epoch',
    logging_steps=5,
    report_to='none',           # wandb 등 외부 로거 비활성
    save_strategy='no',         # 디스크 절약
)

trainer = Trainer(model=clf, args=args, train_dataset=train_ds, eval_dataset=eval_ds,
                  compute_metrics=compute_metrics)

# trainer.train() 은 CPU 에서도 약 1분 내. 실제 fine-tuning 은 GPU 에서.
print('Trainer 객체 준비 완료. 실제 학습은 trainer.train() 호출.')
print('CPU 데모 시 시간이 걸리므로 노트북에서 직접 실행 시에만 풀어 쓰세요.')
# trainer.train()           # ← 직접 실행 시 주석 해제
# trainer.evaluate()


## 8. 한국어 모델 사용 예

영어 외 언어는 영어 BERT 로 잘 안 된다. 한국어용 사전학습 모델이 별도로 있다.

| 모델                 | 용도                | 비고                              |
|----------------------|---------------------|-----------------------------------|
| `klue/bert-base`     | 한국어 NLU 표준     | KLUE 벤치마크와 함께 공개         |
| `klue/roberta-base`  | KLUE 계열 RoBERTa   | 분류/태깅 강세                    |
| `beomi/kcbert-base`  | 댓글 도메인 한국어  | 구어체에 강함                     |
| `monologg/koelectra` | ELECTRA 한국어      | 작고 빠름                         |
| `EleutherAI/polyglot-ko-1.3b` | 한국어 GPT 류 | 텍스트 생성                       |
| `beomi/Llama-3-KoEn-8B` | LLaMA-3 한국어 LM  | LLM 용                            |

> 한국어 모델은 보통 토크나이저 자체가 한국어용으로 학습되어 있어서 `[UNK]` 비율이 훨씬 낮다.


In [ ]:
# 한국어 토크나이저 비교 (영어 모델 vs 한국어 모델)
ko_text = "트랜스포머는 자연어 처리의 표준 아키텍처가 되었다."

print('--- distilbert-base-uncased (영어) ---')
print(tokenizer.tokenize(ko_text))   # 자모 단위로 깨지거나 [UNK] 가 많음

try:
    ko_tok = AutoTokenizer.from_pretrained('klue/bert-base')
    print('\n--- klue/bert-base (한국어) ---')
    print(ko_tok.tokenize(ko_text))
except Exception as e:
    print('klue 모델 로드 실패 (네트워크/캐시 문제일 수 있음):', repr(e))


## 9. encoder-only → decoder-only → instruct-tuned LLM 흐름

같은 transformer 부품으로 만들어진 세 줄기:

```
2017  Transformer (encoder + decoder)         — 번역
       │
       ├─ encoder-only :  BERT, RoBERTa, DistilBERT, KLUE
       │   목적: 입력 표현 학습 (분류, NER, QA 같은 NLU)
       │
       └─ decoder-only :  GPT, GPT-2/3/4, LLaMA, Mistral, Gemma, Polyglot-Ko
           목적: 다음 토큰 생성 (텍스트 생성, 코드 생성, 챗봇)
                ↓
           SFT (지시문 fine-tuning) + RLHF/DPO
                ↓
           ChatGPT, Claude, Gemini 등 instruct-tuned LLM
```

핵심 통찰:
- **아키텍처는 거의 같다.** 다른 건 학습 목표(MLM vs CausalLM), 데이터 규모, 후처리(Instruct/RLHF).
- LLM 이 "새로운 패러다임" 으로 보이지만 안에서 일어나는 연산은 우리가 02 노트북에서 만든 그것과 동일.
- 그래서 "기본" (autograd → MHA → block) 을 알면 LLM 의 행동을 디버깅/이해/수정할 수 있다.


In [ ]:
# 작은 GPT 류로 텍스트 생성 시연 (CPU에서 빠르게)
gen = pipeline('text-generation', model='sshleifer/tiny-gpt2')
print(gen('Once upon a time', max_new_tokens=20, do_sample=True, temperature=0.8)[0]['generated_text'])


## 10. 실전 팁

1. **모델 카드 확인** — `https://huggingface.co/<model_name>` 페이지에 학습 데이터, 한계, 라이선스, 의도된 용도가 있다.
   라이선스 (Apache 2.0, MIT, LLaMA license, Gemma license 등) 는 상업 사용 가능 여부를 좌우.
2. **토크나이저는 모델과 짝** — 다른 모델의 토크나이저로 입력하면 의미 없는 ID 가 만들어진다. 항상 `from_pretrained(MODEL_NAME)` 으로 같이 로드.
3. **`device_map='auto'`** — 큰 모델을 GPU/CPU 에 자동 분산.
4. **양자화** — 8B+ LLM 을 소비자 GPU 에서 돌리려면 `bitsandbytes` 의 4-bit/8-bit, 또는 `gguf`/`AWQ` 양자화.
5. **PEFT/LoRA** — full fine-tuning 대신 추가 어댑터만 학습 (메모리 ↓, 빠름). `peft` 라이브러리.
6. **로컬 추론 옵션** — `llama.cpp` (C++), `vLLM` (서버), `Ollama` (한 줄 실행). HuggingFace 와 별개로 알아두면 유용.
7. **재현성** — `set_seed`, `transformers.set_seed`, `torch.manual_seed`. 단, GPU 비결정성/cuDNN 영향이 있을 수 있음.


## 정리 — 이 학습 시리즈가 닿은 지점

세 폴더를 순서대로 끝낸 학습자는 다음을 모두 "손으로" 만들어 봤다:

```
study_01_pytorch_core   — autograd, 학습 루프 (모든 모델의 기반)
study_02_cnn_rnn        — convolution / recurrence (이전 패러다임의 핵심 두 축)
study_03_transformer    — attention / multi-head / block / 그리고 HuggingFace 로 LLM 까지 연결
```

LLM 시대에도 이 "기본" 이 가치 있는 이유는:

- API 호출 / RAG / 프롬프트 만으로는 디버깅이 안 되는 순간이 온다.
- LoRA/양자화/사전학습 변형/새 아키텍처 제안은 모두 이 기본 위에서 돌아간다.
- LLM 이 짜 준 코드를 "의심하고 검증할 수 있는 능력" 이 차별점이 된다.

이제 다음 단계 후보:
- **PEFT/LoRA 직접 실험** — `peft` 라이브러리로 작은 모델 fine-tuning
- **vLLM / Ollama 로 로컬 LLM 추론** — 인프라 감각
- **RLHF/DPO 의 수식 이해** — instruct-tuning 의 본질
- **새 아키텍처 읽기** — Mamba (SSM), Mixture of Experts, Multimodal transformer
